# Agents in Team

### A single Agent can only say create a short story for us.
### but with a team whre many agents work together towards a common goal they can help us in writing or even helping to review, edit etc.

In [1]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key= os.getenv('OPENAI_API_KEY')
model_client= OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
model_client_2= OpenAIChatCompletionClient(model='gpt-4',api_key=api_key)

In [4]:
from autogen_agentchat.agents import AssistantAgent

dsa_solver= AssistantAgent(
    name='Complex_DSA_Solver',
    model_client=model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words"
)

code_reviewer= AssistantAgent(
    name= 'CODE_REVEIWER',
    model_client=model_client_2,
    description='A Code Reviewer',
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized.Give under 10 words"
)

code_editor= AssistantAgent(
    name= 'CODE_EDITOR',
    model_client=model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required.Give under 10 words"
)

### RoundRobinGroupChat
A team that runs a group chat with participants taking turns in a round-robin fashion to publish a message to all. If a single participant is in the team, the participant will be the only speaker.

https://microsoft.github.io/autogen/stable//reference/python/autogen_agentchat.teams.html#autogen_agentchat.teams.

### Messages
In AutoGen AgentChat, `messages` facilitate communication and information exchange with other agents, orchestrators, and applications. AgentChat supports various message types, each designed for specific purposes.

#### Types of Messages
    1. BaseChatMessage
    2. TextMessage 
    3. MultiModalMessage

https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/tutorial/messages.html

In [5]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage

team= RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    max_turns=15 # -----># maximum number of Message before it stops between the agents.
)

async def test_team():
    task= TextMessage(content='Write a simple Hello world code ?', source='User')
    
    result= await team.run(task=task)

    for each_agent_message in result.messages:
        print(f'{((each_agent_message))} ' )
        print('\n \n')


await test_team()

id='759f84a4-c6ee-47ac-8dcb-8253fafe0f0d' source='User' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 12, 12, 47, 34, 258758, tzinfo=datetime.timezone.utc) content='Write a simple Hello world code ?' type='TextMessage' 

 

id='729d04f7-ca94-4b25-a182-ee2cdf3f58c4' source='Complex_DSA_Solver' models_usage=RequestUsage(prompt_tokens=37, completion_tokens=25) metadata={} created_at=datetime.datetime(2025, 7, 12, 12, 47, 35, 786189, tzinfo=datetime.timezone.utc) content='Certainly! Here\'s a simple Python code to print "Hello, World!":\n\n```python\nprint("Hello, World!")\n```' type='TextMessage' 

 

id='718f0cae-a583-465e-92c0-5967f9c35df0' source='CODE_REVEIWER' models_usage=RequestUsage(prompt_tokens=79, completion_tokens=9) metadata={} created_at=datetime.datetime(2025, 7, 12, 12, 47, 37, 69289, tzinfo=datetime.timezone.utc) content='Code is already optimized, no changes needed.' type='TextMessage' 

 

id='0dcf229c-4dfb-4c09-b503-0bce7aff4f63' source='CODE_EDIT

### Resetting a Team
You can reset the team by calling the `reset()` method. This method will clear the team’s state, including all agents. It will call the each agent’s `on_reset()` method to clear the agent’s state.

In [6]:
await team.reset()  # Reset the team for a new task.
from autogen_agentchat.base import TaskResult

async for message in team.run_stream(task="Write a simple Hello world code ?"):  # type: ignore
    if isinstance(message, TaskResult):
        print("Stop Reason:", message.stop_reason)
    else:
        print(message.source,message)

user id='ee1a2454-263e-4157-82d8-e950cfe2ac83' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 12, 12, 55, 25, 56990, tzinfo=datetime.timezone.utc) content='Write a simple Hello world code ?' type='TextMessage'
Complex_DSA_Solver id='8838e430-9e48-4c04-a7ca-f112fce5a603' source='Complex_DSA_Solver' models_usage=RequestUsage(prompt_tokens=36, completion_tokens=47) metadata={} created_at=datetime.datetime(2025, 7, 12, 12, 55, 26, 301139, tzinfo=datetime.timezone.utc) content='Certainly! Here\'s a simple Python code to print "Hello, World!":\n\n```python\nprint("Hello, World!")\n```\n\nThis code uses the `print()` function to display the message "Hello, World!" on the screen.' type='TextMessage'
CODE_REVEIWER id='2fe11b37-b6d9-4983-be1d-3f03fec199d2' source='CODE_REVEIWER' models_usage=RequestUsage(prompt_tokens=100, completion_tokens=7) metadata={} created_at=datetime.datetime(2025, 7, 12, 12, 55, 27, 845511, tzinfo=datetime.timezone.utc) content='Optimi

In [7]:
from autogen_agentchat.ui import Console
await team.reset()  # Reset the team for a new task.
await Console(team.run_stream(task="Write a simple Hello world code."))  # Stream the messages to the console.


---------- TextMessage (user) ----------
Write a simple Hello world code.


---------- TextMessage (Complex_DSA_Solver) ----------
```python
print("Hello, world!")
```
---------- TextMessage (CODE_REVEIWER) ----------
Code is optimized and concise.
---------- TextMessage (CODE_EDITOR) ----------
Thank you! Glad you found it concise.
---------- TextMessage (Complex_DSA_Solver) ----------
Thank you for the feedback! If you have any other questions or need further assistance, feel free to ask.
---------- TextMessage (CODE_REVEIWER) ----------
Code is already optimized.
---------- TextMessage (CODE_EDITOR) ----------
Great! Let me know if you need help.
---------- TextMessage (Complex_DSA_Solver) ----------
Great to hear! If there's anything else you need, feel free to reach out.
---------- TextMessage (CODE_REVEIWER) ----------
Code is optimal.
---------- TextMessage (CODE_EDITOR) ----------
Thank you for the confirmation!
---------- TextMessage (Complex_DSA_Solver) ----------
Thank you for confirming! If there's anything else you need, just let me know.
--------

TaskResult(messages=[TextMessage(id='51462a31-266f-4168-826a-5d49cbeb1312', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 57, 9, 271518, tzinfo=datetime.timezone.utc), content='Write a simple Hello world code.', type='TextMessage'), TextMessage(id='a9c6c9ec-9d3c-44fa-b83c-8a576988b675', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=36, completion_tokens=10), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 57, 10, 62647, tzinfo=datetime.timezone.utc), content='```python\nprint("Hello, world!")\n```', type='TextMessage'), TextMessage(id='be82749f-9318-4a92-8da2-6f766aa3562f', source='CODE_REVEIWER', models_usage=RequestUsage(prompt_tokens=62, completion_tokens=6), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 57, 10, 915592, tzinfo=datetime.timezone.utc), content='Code is optimized and concise.', type='TextMessage'), TextMessage(id='3077d325-ffdd-42eb-8ef6-13ef9436ed44', source='CODE_EDITOR', mo

In [8]:
await Console(team.run_stream(task="conitnue."))

---------- TextMessage (user) ----------
conitnue.
---------- TextMessage (Complex_DSA_Solver) ----------
Sure! Let me know what you would like to continue with or if there's a specific topic or code you need help with.
---------- TextMessage (CODE_REVEIWER) ----------
Proceeding with the next task.
---------- TextMessage (CODE_EDITOR) ----------
Great! What's the next task?
---------- TextMessage (Complex_DSA_Solver) ----------
Great! Let me know the next task, and I'll be happy to assist you.
---------- TextMessage (CODE_REVEIWER) ----------
Proceeding as instructed.
---------- TextMessage (CODE_EDITOR) ----------
Perfect! What should we tackle next?
---------- TextMessage (Complex_DSA_Solver) ----------
Perfect! Let me know what you'd like to work on next.
---------- TextMessage (CODE_REVEIWER) ----------
Continue as per instructions.
---------- TextMessage (CODE_EDITOR) ----------
Understood! Please provide the next instructions.
---------- TextMessage (Complex_DSA_Solver) --------

TaskResult(messages=[TextMessage(id='052aedfe-8afe-4a7f-8cb7-6de44e4f7e3b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 58, 26, 853334, tzinfo=datetime.timezone.utc), content='conitnue.', type='TextMessage'), TextMessage(id='b11ac164-ce07-49d8-86f0-94c3c1ccef03', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=303, completion_tokens=25), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 58, 28, 103723, tzinfo=datetime.timezone.utc), content="Sure! Let me know what you would like to continue with or if there's a specific topic or code you need help with.", type='TextMessage'), TextMessage(id='dce6be48-5570-41d0-aeb4-f89376515e4b', source='CODE_REVEIWER', models_usage=RequestUsage(prompt_tokens=347, completion_tokens=7), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 58, 29, 37171, tzinfo=datetime.timezone.utc), content='Proceeding with the next task.', type='TextMessage'), TextMessage(id='26e74079

In [9]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number you got from previous run. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number from previous run. Give result as output."
)


In [11]:
my_increment_team= RoundRobinGroupChat(participants=[add_1_agent_first,add_1_agent_second,add_1_agent_third],max_turns=3)

In [12]:
my_increment_team

# Resume a Team

In [13]:
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3


TaskResult(messages=[TextMessage(id='140d3940-10dd-49c1-ae6d-bdf9c09f1e22', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 21, 656319, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='777187cc-7a37-4c43-8a07-1076e04ed744', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 22, 369140, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='6379e6f8-8422-4715-8521-7d38425bfae1', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 23, 3598, tzinfo=datetime.timezone.utc), content='3', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

In [14]:
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
4
---------- TextMessage (add_1_agent_second) ----------
5
---------- TextMessage (add_1_agent_third) ----------
6


TaskResult(messages=[TextMessage(id='9275442a-6c99-4ce4-bcc4-383f0d231634', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=50, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 38, 53150, tzinfo=datetime.timezone.utc), content='4', type='TextMessage'), TextMessage(id='a989ba41-76e8-45b6-b843-41abb097b976', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=60, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 39, 61564, tzinfo=datetime.timezone.utc), content='5', type='TextMessage'), TextMessage(id='be69f762-45e5-4e40-8532-9b884c1bfaaf', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=67, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 39, 639462, tzinfo=datetime.timezone.utc), content='6', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

In [15]:
await team.reset()
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
7
---------- TextMessage (add_1_agent_second) ----------
8
---------- TextMessage (add_1_agent_third) ----------
9


TaskResult(messages=[TextMessage(id='bf5a7228-4d46-4139-85d9-8c06e56b970a', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=76, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 47, 56604, tzinfo=datetime.timezone.utc), content='7', type='TextMessage'), TextMessage(id='089535f3-540e-4eb2-95fb-01779d90f86e', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=86, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 47, 626717, tzinfo=datetime.timezone.utc), content='8', type='TextMessage'), TextMessage(id='2a1138d2-b638-4f9a-9140-015d7562ddc4', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=92, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 12, 12, 59, 48, 147658, tzinfo=datetime.timezone.utc), content='9', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

In [16]:
await my_increment_team.reset()
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
The result of adding 1 to the number 0 is 1.
---------- TextMessage (add_1_agent_second) ----------
The result of adding 1 to the number 1 is 2.
---------- TextMessage (add_1_agent_third) ----------
The result of adding 1 to the number 2 is 3.


TaskResult(messages=[TextMessage(id='ccc1b005-e4ca-4e89-ae60-090bc9be5979', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 7, 12, 13, 0, 5, 177986, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 0 is 1.', type='TextMessage'), TextMessage(id='af03cf87-62f1-4423-9d90-2ce23973bdc3', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=48, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 7, 12, 13, 0, 5, 900399, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 1 is 2.', type='TextMessage'), TextMessage(id='d07febea-5af2-43f9-af4c-1513c7a1b9bc', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=70, completion_tokens=15), metadata={}, created_at=datetime.datetime(2025, 7, 12, 13, 0, 6, 597876, tzinfo=datetime.timezone.utc), content='The result of adding 1 to the number 2 is 3.', type='